# Week 4 exercise — code generator (Python -> C++)

Use frontier models to rewrite slow Python as high-performance C++, then **compile and
benchmark** to prove the speed-up. Two models generate the C++ (`gpt-4o-mini` vs
`claude-haiku-4.5`) so we can compare them — the week-4 theme of *picking the right model*.

In [1]:
import os, shutil, subprocess, time
import anthropic
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv(override=True)
frontier = OpenAI()
claude = anthropic.Anthropic()

# g++ from PATH; fallback to the choco-installed MinGW (space-free path)
GPP = shutil.which("g++") or r"C:\ProgramData\mingw64\mingw64\bin\g++.exe"
BUILD = r"C:\Users\Public\llm_week4_build"
os.makedirs(BUILD, exist_ok=True)
print("g++:", GPP)

g++: C:\ProgramData\mingw64\mingw64\bin\g++.exe


In [2]:
# The slow Python we want to accelerate (integer-only so C++ output matches exactly)
PYTHON = '''
def count_primes(n):
    count = 0
    for x in range(2, n):
        d, prime = 2, True
        while d * d <= x:
            if x % d == 0:
                prime = False
                break
            d += 1
        if prime:
            count += 1
    return count

print(count_primes(300000))
'''

In [3]:
SYSTEM = (
    "Rewrite the given Python as a single high-performance C++17 program. "
    "Reply with ONLY C++ source code - no markdown, no commentary. Include all headers "
    "and an int main() that prints exactly the same output as the Python."
)

def clean(text):
    return "\n".join(l for l in text.splitlines() if not l.strip().startswith("```")).strip()

def to_cpp(provider, model):
    if provider == "anthropic":
        msg = claude.messages.create(model=model, system=SYSTEM, max_tokens=1500,
                                     messages=[{"role": "user", "content": PYTHON}])
        return clean(msg.content[0].text)
    r = frontier.chat.completions.create(model=model, messages=[
        {"role": "system", "content": SYSTEM}, {"role": "user", "content": PYTHON}])
    return clean(r.choices[0].message.content)

def compile_and_run(cpp, name):
    src, exe = os.path.join(BUILD, name + ".cpp"), os.path.join(BUILD, name + ".exe")
    with open(src, "w") as f:
        f.write(cpp)
    subprocess.run([GPP, "-O3", "-std=c++17", "-o", exe, src], check=True, capture_output=True, text=True)
    t = time.time()
    out = subprocess.run([exe], capture_output=True, text=True).stdout.strip()
    return out, time.time() - t

### Baseline: run the Python

In [4]:
ns = {}
t = time.time(); exec(PYTHON, ns); py_time = time.time() - t
print(f"python: {py_time:.3f}s")

25997
python: 0.996s


### Generate C++ with each model, compile, run, compare

In [5]:
for label, provider, model in [
    ("gpt-4o-mini", "openai", "gpt-4o-mini"),
    ("claude-haiku-4.5", "anthropic", "claude-haiku-4-5-20251001"),
]:
    cpp = to_cpp(provider, model)
    out, cpp_time = compile_and_run(cpp, label.replace(".", ""))
    speedup = py_time / cpp_time if cpp_time else float("inf")
    print(f"{label:<18} output={out:<8} time={cpp_time:.4f}s  speedup x{speedup:,.0f}")

gpt-4o-mini        output=25997    time=0.0710s  speedup x14


claude-haiku-4.5   output=25997    time=0.0715s  speedup x14
